In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)


np.random.seed(42)
print("setup complete")

setup complete


In [4]:
df = pd.read_csv('../data/cs-training.csv', index_col=0)
original_shape =df.shape
print(f"Original dataset shape: {original_shape}")
print(f"Original memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

Original dataset shape: (150000, 11)
Original memory usage: 12.59 MB


In [5]:
delinquency_cols=[
    'NumberOfTime30-59DaysPastDueNotWorse',
    'NumberOfTime60-89DaysPastDueNotWorse',
    'NumberOfTimes90DaysLate'
]

for col in delinquency_cols:

    print(f"\n=== {col} ===")
    print(df[col].value_counts().sort_index().tail(10))


=== NumberOfTime30-59DaysPastDueNotWorse ===
NumberOfTime30-59DaysPastDueNotWorse
6     140
7      54
8      25
9      12
10      4
11      1
12      2
13      1
96      5
98    264
Name: count, dtype: int64

=== NumberOfTime60-89DaysPastDueNotWorse ===
NumberOfTime60-89DaysPastDueNotWorse
3     318
4     105
5      34
6      16
7       9
8       2
9       1
11      1
96      5
98    264
Name: count, dtype: int64

=== NumberOfTimes90DaysLate ===
NumberOfTimes90DaysLate
9      19
10      8
11      5
12      2
13      4
14      2
15      2
17      1
96      5
98    264
Name: count, dtype: int64


In [ ]:
mask_30=df['NumberOfTime30-59DaysPastDueNotWorse'].isin([96,98])
mask_60=df['NumberOfTime60-89DaysPastDueNotWorse'].isin([96,98])
mask_90=df['NumberOfTimes90DaysLate'].isin([96,98])
# Har bir mask'da nechta True bor — sanab ko'ramiz
print(f"30-59 ustunda 96 yoki 98 bo'lganlar: {mask_30.sum()}")
print(f"60-89 ustunda 96 yoki 98 bo'lganlar: {mask_60.sum()}")
print(f"90 ustunda 96 yoki 98 bo'lganlar: {mask_90.sum()}")


print(f"\nUchchalasida ham 96/98: {(mask_30 & mask_60 & mask_90).sum()}")
print(f"Faqat 30-59'da: {(mask_30 & ~mask_60 & ~mask_90).sum()}")

30-59 ustunda 96 yoki 98 bo'lganlar: 269
60-89 ustunda 96 yoki 98 bo'lganlar: 269
90 ustunda 96 yoki 98 bo'lganlar: 269

Uchchalasida ham 96/98: 269
Faqat 30-59'da: 0


In [8]:
# 96-98 maxsus kodlarini topish (3 ta delinquency ustunida)
# Avvalgi tahlilda biz tasdiqladik: uchala ustunda ham bir xil 269 odam
mask_invalid = (
    df['NumberOfTime30-59DaysPastDueNotWorse'].isin([96, 98]) |
    df['NumberOfTime60-89DaysPastDueNotWorse'].isin([96, 98]) |
    df['NumberOfTimes90DaysLate'].isin([96, 98])
)

# O'chirishdan oldin va keyin shape'ni taqqoslaymiz
print(f"O'chirishdan oldin: {df.shape}")
print(f"O'chiriladigan qatorlar: {mask_invalid.sum()}")

# Mask teskari qilib filtering — faqat valid qatorlarni qoldiramiz
# ~mask_invalid = mask_invalid'ning teskarisi
df = df[~mask_invalid]

# Index'larni qaytadan tartiblash (qator o'chirilgandan keyin nomerlar buziladi)
# .reset_index(drop=True) — yangi 0, 1, 2, ... index yaratadi
# drop=True — eski index'ni ustun qilib saqlamaslik
df = df.reset_index(drop=True)

print(f"O'chirishdan keyin: {df.shape}")
print(f"Yo'qolgan foiz: {(269 / 150000) * 100:.2f}%")

O'chirishdan oldin: (150000, 11)
O'chiriladigan qatorlar: 269
O'chirishdan keyin: (149731, 11)
Yo'qolgan foiz: 0.18%


In [9]:
# Yoshi 0 bo'lgan qatorni topish va o'chirish
# Avval qancha borligini tekshiramiz
age_zero_count = (df['age'] == 0).sum()
print(f"Yoshi 0 bo'lganlar: {age_zero_count}")

# O'chirish (faqat 1 ta qator bo'lganligi uchun xavfsiz)
df = df[df['age'] > 0]
df = df.reset_index(drop=True)

print(f"O'chirishdan keyin shape: {df.shape}")

Yoshi 0 bo'lganlar: 1
O'chirishdan keyin shape: (149730, 11)


In [10]:
# Yosh outlier'larni o'chirish (100 dan katta yosh — shubhali)
# Bu business qaror: 100+ yoshli kredit oluvchilar real bo'lsa-da,
# 22 ta odam 99-107 oraliqda bo'lishi data anomaliyasi belgisi

# Avval qancha qator o'chirilishini ko'ramiz
age_outlier_count = (df['age'] > 100).sum()
print(f"Yoshi 100 dan katta: {age_outlier_count}")

# Whitelist filtering: faqat 0 < age <= 100 oraliqdagi qatorlar
# Bu chegara — business assumption (suhbatda tushuntiraman)
df = df[df['age'] <= 100]
df = df.reset_index(drop=True)

print(f"Shape: {df.shape}")
print(f"Min age: {df['age'].min()}, Max age: {df['age'].max()}")

Yoshi 100 dan katta: 13
Shape: (149717, 11)
Min age: 21, Max age: 99


In [12]:
missing= df.isnull().sum()
missing_pct= (df.isnull().sum()/len(df))*100

missing_df=pd.DataFrame({
    'Missing Count' :missing,
    'Missing %': missing_pct.round(2)

})
print(missing_df[missing_df['Missing Count']> 0])
print(f"\n Jami qatorlar: {len(df)}")

                    Missing Count  Missing %
MonthlyIncome               29603      19.77
NumberOfDependents           3890       2.60

 Jami qatorlar: 149717


In [13]:
# 1-qadam: Flag yaratish — BEFORE imputation
# Diqqat: flag'ni median bilan to'ldirishdan OLDIN yaratamiz
# Aks holda hamma qatorda flag=0 bo'lib qoladi (chunki null'lar yo'q bo'lib ketgan)
df['income_missing_flag'] = df['MonthlyIncome'].isnull().astype(int)

# .astype(int) — True/False qiymatlarni 1/0 ga aylantirish
# Bu ML model uchun muhim: model boolean emas, son qabul qiladi

# 2-qadam: Median qiymatni hisoblash (null'larni hisobga olmasdan)
# .median() avtomatik null'larni o'tkazib yuboradi
income_median = df['MonthlyIncome'].median()
print(f"MonthlyIncome median: {income_median}")

# 3-qadam: Null'larni median bilan to'ldirish
# .fillna(value) — null qiymatlarni berilgan qiymat bilan almashtirish
df['MonthlyIncome'] = df['MonthlyIncome'].fillna(income_median)

# 4-qadam: Tekshirish — null qolmadimi?
print(f"\nMonthlyIncome null qolgan: {df['MonthlyIncome'].isnull().sum()}")

# 5-qadam: Flag taqsimoti — nechta odamda missing edi
print(f"\nIncome missing flag taqsimoti:")
print(df['income_missing_flag'].value_counts())
print(f"\nFoiz: {(df['income_missing_flag'].sum() / len(df) * 100):.2f}%")

MonthlyIncome median: 5400.0

MonthlyIncome null qolgan: 0

Income missing flag taqsimoti:
income_missing_flag
0    120114
1     29603
Name: count, dtype: int64

Foiz: 19.77%


In [14]:
# NumberOfDependents null'larini 0 bilan to'ldirish
# Mantiq: median ham 0, demak ko'pincha null = yolg'iz odam

# Avval median'ni tasdiqlash (sabab uchun)
dependents_median = df['NumberOfDependents'].median()
print(f"NumberOfDependents median: {dependents_median}")

# Null'larni 0 bilan to'ldirish
df['NumberOfDependents'] = df['NumberOfDependents'].fillna(0)

# Bonus: float'dan int'ga aylantirish
# NumberOfDependents — bu BOLA SONI, decimal bo'lishi mumkin emas
# Hozir u float64 (chunki null'lar bor edi, pandas avtomatik float qiladi)
# Endi null'lar yo'q, int'ga aylantiramiz
df['NumberOfDependents'] = df['NumberOfDependents'].astype(int)

# Tekshirish
print(f"\nNull qolgan: {df['NumberOfDependents'].isnull().sum()}")
print(f"Dtype: {df['NumberOfDependents'].dtype}")
print(f"\nTaqsimot:")
print(df['NumberOfDependents'].value_counts().sort_index())

NumberOfDependents median: 0.0

Null qolgan: 0
Dtype: int64

Taqsimot:
NumberOfDependents
0     90595
1     26292
2     19501
3      9479
4      2860
5       745
6       158
7        51
8        24
9         5
10        5
13        1
20        1
Name: count, dtype: int64


In [15]:
# To'liq tekshirish — missing qolmadimi?
print("=== MISSING VALUES SUMMARY ===")
print(df.isnull().sum())
print(f"\nJami null: {df.isnull().sum().sum()}")
print(f"Jami qatorlar: {len(df)}")
print(f"Ustunlar: {df.shape[1]}")

=== MISSING VALUES SUMMARY ===
SeriousDlqin2yrs                        0
RevolvingUtilizationOfUnsecuredLines    0
age                                     0
NumberOfTime30-59DaysPastDueNotWorse    0
DebtRatio                               0
MonthlyIncome                           0
NumberOfOpenCreditLinesAndLoans         0
NumberOfTimes90DaysLate                 0
NumberRealEstateLoansOrLines            0
NumberOfTime60-89DaysPastDueNotWorse    0
NumberOfDependents                      0
income_missing_flag                     0
dtype: int64

Jami null: 0
Jami qatorlar: 149717
Ustunlar: 12


In [16]:
# DebtRatio hozirgi holati — cleaning'dan keyin
# Boshida: max = 329,664. Hozir ham shunday qoldimi?

print(f"DebtRatio statistikasi (hozirgi):")
print(f"  Min:    {df['DebtRatio'].min()}")
print(f"  Max:    {df['DebtRatio'].max()}")
print(f"  Mean:   {df['DebtRatio'].mean():.2f}")
print(f"  Median: {df['DebtRatio'].median():.4f}")

# Income_missing_flag bo'yicha alohida-alohida ko'ramiz
print(f"\n--- Income normal odamlar (flag=0) ---")
normal_debt = df[df['income_missing_flag'] == 0]['DebtRatio']
print(f"  Median: {normal_debt.median():.4f}")
print(f"  Max:    {normal_debt.max():.2f}")
print(f"  > 1 foizi: {(normal_debt > 1).sum() / len(normal_debt) * 100:.2f}%")

print(f"\n--- Income imputed odamlar (flag=1) ---")
imputed_debt = df[df['income_missing_flag'] == 1]['DebtRatio']
print(f"  Median: {imputed_debt.median():.4f}")
print(f"  Max:    {imputed_debt.max():.2f}")
print(f"  > 1 foizi: {(imputed_debt > 1).sum() / len(imputed_debt) * 100:.2f}%")

DebtRatio statistikasi (hozirgi):
  Min:    0.0
  Max:    329664.0
  Mean:   353.63
  Median: 0.3671

--- Income normal odamlar (flag=0) ---
  Median: 0.2964
  Max:    61106.50
  > 1 foizi: 6.02%

--- Income imputed odamlar (flag=1) ---
  Median: 1171.0000
  Max:    329664.00
  > 1 foizi: 94.15%


In [17]:
# Normal guruhning median DebtRatio'sini hisoblaymiz
# Faqat flag=0 odamlardan (haqiqiy nisbat bor)
normal_debt_median = df[df['income_missing_flag'] == 0]['DebtRatio'].median()
print(f"Normal DebtRatio median: {normal_debt_median:.4f}")

# Imputed income guruhda (flag=1) DebtRatio'ni median bilan almashtiramiz
# .loc[row_condition, column] — aniq qator-ustun yacheykalarni o'zgartirish
df.loc[df['income_missing_flag'] == 1, 'DebtRatio'] = normal_debt_median

# Tekshirish
print(f"\nDebtRatio statistikasi (tuzatilgandan keyin):")
print(f"  Min:    {df['DebtRatio'].min()}")
print(f"  Max:    {df['DebtRatio'].max()}")
print(f"  Mean:   {df['DebtRatio'].mean():.2f}")
print(f"  Median: {df['DebtRatio'].median():.4f}")

# Qayta tahlil — guruhlar bo'yicha
print(f"\nflag=0 max: {df[df['income_missing_flag'] == 0]['DebtRatio'].max():.2f}")
print(f"flag=1 max: {df[df['income_missing_flag'] == 1]['DebtRatio'].max():.2f}")



Normal DebtRatio median: 0.2964

DebtRatio statistikasi (tuzatilgandan keyin):
  Min:    0.0
  Max:    61106.5
  Mean:   21.41
  Median: 0.2964

flag=0 max: 61106.50
flag=1 max: 0.30
